In [6]:
import pandas as pd
import numpy as np

# 1. Load data
df_emb = pd.read_csv('embedding_bertopic_v2.csv')              # 'Game', 'cleaned_Reviews', embedding_*
df_asg = pd.read_csv('hasil_topic_assignment_automerged.csv')  # 'game', 'cleaned_reviews', 'topic', UMAP_*, dll.

# 2. Merge
df = pd.merge(
    df_emb,
    df_asg[['cleaned_Reviews', 'topic']],
    on='cleaned_Reviews',
    how='inner'
)
df = df.drop(columns=['cleaned_Reviews'])

# 3. Kolom embedding
emb_cols = [c for c in df.columns if c.startswith('embedding_')]

# 4. Hitung centroid per topic (kecuali -1)
centroids = df[df['topic'] != -1].groupby('topic')[emb_cols].mean()
centroids = centroids.div(np.linalg.norm(centroids, axis=1), axis=0)

# 5. Normalisasi semua embedding review
emb_norm = df[emb_cols].div(np.linalg.norm(df[emb_cols], axis=1), axis=0).values

# 6. Hitung similarity ke semua centroid
sim_matrix = np.dot(emb_norm, centroids.values.T)  # shape: (n_reviews, n_topics)
topic_labels = centroids.index.to_list()

# 7. Ambil topik dominan untuk setiap Game
df_sim = pd.DataFrame(sim_matrix, columns=topic_labels)
df_sim['Game'] = df['Game']

# 8. Ambil rata-rata similarity per game-topic
gt_sim = df_sim.melt(id_vars='Game', var_name='topic', value_name='similarity')
gt_mean = gt_sim.groupby(['Game', 'topic'])['similarity'].mean().reset_index()

# 9. Tambahkan penalti untuk topik yang terlalu umum (misal topik 0)
topic_counts = gt_mean['topic'].value_counts(normalize=True)
penalty_factor = 1 - topic_counts  # topik yang sering muncul → penalti lebih besar
gt_mean['penalized_sim'] = gt_mean.apply(
    lambda r: r['similarity'] * penalty_factor.get(r['topic'], 1),
    axis=1
)

# 10. Pilih topic dengan penalized_sim tertinggi
idx = gt_mean.groupby('Game')['penalized_sim'].idxmax()
dominant = gt_mean.loc[idx, ['Game', 'topic', 'similarity']].reset_index(drop=True)

# 11. Rename kolom
dominant = dominant.rename(columns={
    'topic': 'Dominant_Topic',
    'similarity': 'Similarity_Score'
})

# 12. Simpan hasil
dominant.to_csv('dominant_topic_per_game.csv', index=False)
print(dominant.head())


                   Game  Dominant_Topic  Similarity_Score
0  (the) Gnorp Apologue               0          0.719435
1    ---Red---Tether-->               0          0.650721
2           .Forty-Five               0          0.594609
3    10 Miles To Safety               0          0.659972
4  10 Minutes Till Dawn               2          0.533373


# Tidak ambil topik 0

In [8]:
import pandas as pd
import numpy as np
from collections import Counter

# 1. Load data
df_dom = pd.read_csv("dominant_topic_per_game.csv")   # Game, Dominant_Topic, Similarity_Score
df_seg = pd.read_csv("cleaned_with_genre_codes.csv")     # Steam ID, Game Name, Playtime (hours), Genres, Achievements

# 2. Standardisasi nama game agar cocok
df_seg["Game Name"] = df_seg["Game Name"].str.strip().str.lower()
df_dom["Game"]       = df_dom["Game"].str.strip().str.lower()

# 3. Merge untuk mendapatkan Dominant_Topic per baris game–pemain
merged = pd.merge(
    df_seg, df_dom[["Game","Dominant_Topic"]],
    left_on="Game Name", right_on="Game",
    how="inner"
)

# 4. Filter invalid dan hanya pemain yang main ≥10 menit
merged = merged[merged["Game Name"] != "unknown"]
merged = merged[merged["Playtime (hours)"] >= 0.167]

# 5. Buang NaN & duplikat
merged.dropna(subset=[
    "Steam ID","Game Name","Playtime (hours)",
    "Genres","Achievements","Dominant_Topic"
], inplace=True)
merged.drop_duplicates(subset=["Steam ID","Game Name"], inplace=True)

# 6. Hanya pemain aktif (>= kuartil 1 total game)
game_counts = (
    merged
    .groupby("Steam ID")["Game Name"]
    .nunique()
    .reset_index(name="Total_Games")
)
q1 = game_counts["Total_Games"].quantile(0.25)
active_ids = game_counts[game_counts["Total_Games"] >= q1]["Steam ID"]
merged = merged[merged["Steam ID"].isin(active_ids)]

# 7. Hitung Total_Achievements & Avg_Playtime
achievement_sum = (
    merged
    .groupby("Steam ID")["Achievements"]
    .sum()
    .reset_index(name="Total_Achievements")
)
playtime_avg = (
    merged
    .groupby("Steam ID")["Playtime (hours)"]
    .mean()
    .reset_index(name="Avg_Playtime")
)

# 8. Mapping Genres → kode numerik
merged["Genres"] = merged["Genres"].astype(str)
genre_list = sorted({g for gs in merged["Genres"] for g in gs.split(", ")})
genre_map  = {g:i+1 for i,g in enumerate(genre_list)}
merged["Genre_Code"] = merged["Genres"].apply(
    lambda x: [genre_map[g] for g in x.split(", ") if g in genre_map]
)

# 9. Buat fungsi top_n dan ambil Top 3 Genre per pemain
def top_n(seq, n=3):
    c = Counter(seq)
    return [item for item, _ in c.most_common(n)]

top_genres = (
    merged
    .groupby("Steam ID")["Genre_Code"]
    .apply(lambda lists: top_n([g for sub in lists for g in sub], 3))
    .reset_index(name="Top_3_Genres")
)

# 10. Ambil Dominant_Topic bersih per pemain
# Skip topics -1 (noise) and 0
topic_lists = (
    merged
    .groupby("Steam ID")["Dominant_Topic"]
    .apply(lambda ts: [t for t,_ in Counter(ts).most_common()])
)
def choose_topic(lst):
    for t in lst:
        if t not in [-1, 0]:  # skip noise and topic 0
            return t
    return np.nan

dominant_topic = (
    topic_lists
    .apply(choose_topic)
    .reset_index(name="Topic Dominan")
)

# 11. Gabungkan semua ke summary
summary = (
    game_counts
    .merge(achievement_sum, on="Steam ID")
    .merge(playtime_avg,    on="Steam ID")
    .merge(dominant_topic,  on="Steam ID")
    .merge(top_genres,      on="Steam ID")
)

# 12. Explode Top_3_Genres → satu baris per genre dominan
exploded = (
    summary
    .explode("Top_3_Genres")
    .rename(columns={
        "Steam ID":         "Steam ID",
        "Total_Games":      "Total Game",
        "Avg_Playtime":     "Total Playtime",
        "Total_Achievements":"Total Achievement",
        "Top_3_Genres":     "Genre Dominan"
    })
)

# 13. Tambah kolom No
exploded.insert(0, "No", range(1, len(exploded) + 1))

# 14. Simpan hasil
exploded.to_csv("transformation_segmentation_v4.csv", index=False)
print("✅ Selesai. Hasil dengan Top 3 Genre explode tersimpan di 'transformation_segmentation_v4.csv'")


✅ Selesai. Hasil dengan Top 3 Genre explode tersimpan di 'transformation_segmentation_v4.csv'


In [5]:
# Simpan mapping genre dan kode
genre_df = pd.DataFrame(list(genre_code_map.items()), columns=["Genre", "Genre_Code"])
genre_df.to_csv("genre_code_mapping.csv", index=False)
print("✅ Mapping kode genre disimpan di 'genre_code_mapping.csv'")


NameError: name 'genre_code_map' is not defined